# Exploratory Data Analysis: Olist Dataset
This notebook loads the 9 Olist datasets, explores their structures, merges them, and performs visual and statistical analyses.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
# %matplotlib inline

## 1. Load Data
We load all 9 CSV files and display their shapes, dtypes, and null counts.

In [ ]:
data_dir = '../dataset olist/'

files = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'translations': 'product_category_name_translation.csv'
}

dfs = {}
for name, file in files.items():
    dfs[name] = pd.read_csv(os.path.join(data_dir, file))
    print(f"--- {name.upper()} ---")
    print(f"Shape: {dfs[name].shape}")
    print(f"Nulls: {dfs[name].isnull().sum().sum()}")
    print("-" * 30)

## 2. Merge Data
Merging datasets into a single master dataframe.

In [ ]:
# Merging strategy
df = dfs['order_items'].merge(dfs['orders'], on='order_id', how='left')
df = df.merge(dfs['products'], on='product_id', how='left')
df = df.merge(dfs['translations'], on='product_category_name', how='left')
df = df.merge(dfs['order_reviews'].drop_duplicates(subset=['order_id']), on='order_id', how='left')
df = df.merge(dfs['customers'], on='customer_id', how='left')
df = df.merge(dfs['sellers'], on='seller_id', how='left')
df = df.merge(dfs['order_payments'].groupby('order_id')['payment_value'].sum().reset_index(), on='order_id', how='left')

# Use english category names
df['category'] = df['product_category_name_english'].fillna(df['product_category_name'])

# Convert timestamps
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print(f"Master dataframe shape: {df.shape}")

## 3. Visualizations
- Order volume by month (2016-2018)
- Top 20 product categories by revenue
- Price distribution (log scale)
- Review score distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Order volume by month
df['year_month'] = df['order_purchase_timestamp'].dt.to_period('M')
monthly_orders = df.groupby('year_month')['order_id'].nunique()
monthly_orders.plot(kind='bar', ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Order Volume by Month (2016-2018)')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Number of Orders')

# Top 20 categories by revenue
category_revenue = df.groupby('category')['price'].sum().sort_values(ascending=False).head(20)
category_revenue.plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Top 20 Categories by Revenue')
axes[0, 1].set_xlabel('Category')
axes[0, 1].set_ylabel('Revenue')

# Price distribution (log scale)
np.log1p(df['price']).plot(kind='hist', bins=50, ax=axes[1, 0], color='green', edgecolor='black')
axes[1, 0].set_title('Price Distribution (Log Scale)')
axes[1, 0].set_xlabel('Log(1 + Price)')

# Review score distribution
df['review_score'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Review Score Distribution')
axes[1, 1].set_xlabel('Review Score')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Metrics & Correlation
- Avg price per category
- Demand velocity (orders per day per SKU)
- Price-review correlation heatmap

In [ ]:
# Avg price per category
avg_price_cat = df.groupby('category')['price'].mean().sort_values(ascending=False)
print("Top 5 Categories by Avg Price:")
print(avg_price_cat.head())

# Demand velocity (orders per day per SKU)
total_days = (df['order_purchase_timestamp'].max() - df['order_purchase_timestamp'].min()).days
sku_orders = df.groupby('product_id')['order_id'].nunique()
demand_velocity = sku_orders / total_days
print("\nTop 5 SKUs by Demand Velocity (Orders/Day):")
print(demand_velocity.sort_values(ascending=False).head())

# Price-review correlation
corr_df = df[['price', 'review_score', 'freight_value']].dropna()
plt.figure(figsize=(6, 4))
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Price vs Review Score Correlation')
plt.show()

## 5. Data Quality Issues
- Duplicate order_ids
- SKUs with < 5 orders
- Price outliers (> 3 std devs)

In [ ]:
# Duplicates
dup_orders = df['order_id'].duplicated().sum()
print(f"Duplicate order_items rows (expected for multi-item orders): {dup_orders}")

# SKUs with < 5 orders
sku_counts = df['product_id'].value_counts()
low_volume_skus = (sku_counts < 5).sum()
print(f"SKUs with < 5 orders: {low_volume_skus} out of {len(sku_counts)} ({low_volume_skus/len(sku_counts):.1%})")

# Price outliers (> 3 std devs)
price_mean = df['price'].mean()
price_std = df['price'].std()
outliers = df[df['price'] > (price_mean + 3 * price_std)]
print(f"Price outliers (> 3 std devs): {len(outliers)} items")